In [1]:
from ogx_client import OgxClient
import rich

In [2]:
# Configuration
OGX_CONNECTION_URL = "http://ogxserver-service.llama.svc.cluster.local:8321"

In [3]:
# Initialize OGX client
client = OgxClient(base_url=OGX_CONNECTION_URL)

In [4]:
# List available models
models = client.models.list()
rich.print(models)

ListModelsResponse(
    data=[
        Model(
            id='vllm-embedding/granite-embeddings',
            created=1780909574,
            owned_by='ogx',
            custom_metadata={
                'model_type': 'embedding',
                'provider_id': 'vllm-embedding',
                'provider_resource_id': 'granite-embeddings',
                'embedding_dimension': 768
            },
            object='model'
        ),
        Model(
            id='vllm-inference/llama-32-3b-instruct',
            created=1780909574,
            owned_by='ogx',
            custom_metadata={
                'model_type': 'llm',
                'provider_id': 'vllm-inference',
                'provider_resource_id': 'llama-32-3b-instruct'
            },
            object='model'
        )
    ],
    object='list'
)

In [5]:
response = client.chat.completions.create(
    model="vllm-inference/llama-32-3b-instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about open source"},
    ],
)
print(response.choices[0].message.content)	

Freedom in the code
Shared knowledge, hearts united
Community born


In [6]:
response = client.with_options(timeout=600.0).responses.create(
    model="vllm-inference/llama-32-3b-instruct",
    input="What is the capital of India?"
)
rich.print(response.output_text)

The capital of India is New Delhi.

In [7]:
response = client.with_options(timeout=600.0).responses.create(
    model="vllm-inference/llama-32-3b-instruct",
    input="What is 12 multiplied by 7?",
    tools=[{
        "type": "mcp",
        "server_label": "MathOperationsServer",
        "server_url": "http://math-mcp-server.llama.svc.cluster.local:9000/sse", # "http://host.containers.internal:9000/sse",
    }],
)

In [8]:
# Checking if any vector db present
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](data=[], has_more=False, last_id='', object='list', first_id='')

In [9]:
# Extract LLM and embedding model details
llm_model = next(
    m for m in models.data
    if m.custom_metadata.get("model_type") == "llm"
)

# Using specifically sentence-transformers because customized the config to use this inline model
embedding_model = next(
    m for m in models.data
    if m.custom_metadata.get("model_type") == "embedding"
)

model_id = llm_model.id
embedding_model_id = embedding_model.id

print(f"LLM Model: {model_id}")
print(f"Embedding Model: {embedding_model_id}")

embedding_dimension = embedding_model.custom_metadata["embedding_dimension"]
print(f"Embedding Dimension: {embedding_dimension}")


LLM Model: vllm-inference/llama-32-3b-instruct
Embedding Model: vllm-embedding/granite-embeddings
Embedding Dimension: 768


In [10]:
# Create vector store with pgvector
vector_store = client.vector_stores.create(
    name="techmart_policy_store",
    extra_body={
        "embedding_model": embedding_model_id,
        "embedding_dimension": embedding_dimension,
        "provider_id": "pgvector"
    },
)

In [11]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_5fdffdec-ec52-4218-b0d5-1936b96f8a34',
            created_at=1780909634,
            file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1780909634,
            metadata={
                'provider_id': 'pgvector',
                'provider_vector_store_id': 'vs_5fdffdec-ec52-4218-b0d5-1936b96f8a34',
                'embedding_model': 'vllm-embedding/granite-embeddings',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_5fdffdec-ec52-4218-b0d5-1936b96f8a34',
    object='list',
    first_id='vs_5fdffdec-ec52-4218-b0d5-1936b96f8a34'
)

In [13]:
# Policy file
POLICY_FILE = "data/return-policy.txt"

# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info = client.files.create(

        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info.id}")

Uploaded file: file-2a7bfe16e1724e35b54ffa0b4bca9686


In [14]:
vector_store_id = vector_store.id

# Add file to vector store with chunking strategy
vector_store_file = client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_info.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 400,
            "chunk_overlap_tokens": 100,
        },
    },
)

rich.print(vector_store_file)

VectorStoreFile(
    id='file-2a7bfe16e1724e35b54ffa0b4bca9686',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1780909672,
    status='completed',
    vector_store_id='vs_5fdffdec-ec52-4218-b0d5-1936b96f8a34',
    attributes={},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [15]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_5fdffdec-ec52-4218-b0d5-1936b96f8a34',
            created_at=1780909634,
            file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1780909634,
            metadata={
                'provider_id': 'pgvector',
                'provider_vector_store_id': 'vs_5fdffdec-ec52-4218-b0d5-1936b96f8a34',
                'embedding_model': 'vllm-embedding/granite-embeddings',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_5fdffdec-ec52-4218-b0d5-1936b96f8a34',
    object='list',
    first_id='vs_5fdffdec-ec52-4218-b0d5-1936b96f8a34'
)

In [19]:
INSTRUCTIONS_PROMPT = """You are a helpful and professional customer service assistant.

WORKFLOW:
1. Analyze the customer's question carefully
2. Use available tools to gather all necessary information
3. After gathering information, provide a COMPLETE, well-structured answer

RESPONSE REQUIREMENTS:
- Be clear, accurate, and professional
- Include all relevant details from the tools
- Structure your answer logically
- Provide actionable next steps when applicable
- Never stop after calling tools - always synthesize the final answer

IMPORTANT: You MUST provide a final answer after using tools. Be helpful, accurate, and thorough."""


In [21]:
# Test 1: General return policy question
query = "What is the return window for electronics in tech mart?"

response = client.with_options(timeout=600.0).responses.create(
    model=model_id,
    input=query,
    instructions=INSTRUCTIONS_PROMPT,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store_id],
        }
    ],
)

print("\n" + "="*80)
print(f"QUESTION: {query}")
print("="*80)
print(f"\nANSWER:\n{response.output_text}")
print("\n" + "="*80)


QUESTION: What is the return window for electronics in tech mart?

ANSWER:
The return window for electronics at Tech Mart is 15 days from the date of delivery.



In [22]:
rich.print(response)

ResponseObject(
    id='resp_2d3a142f-685b-49d1-aa0a-d1ac9b3256da',
    created_at=1780915895,
    model='vllm-inference/llama-32-3b-instruct',
    output=[
        OutputOpenAIResponseOutputMessageFileSearchToolCall(
            id='fc_bb60f8b2-27b6-4cc0-8017-0e9b4a7d3a74',
            queries=['Tech Mart electronics return window'],
            status='completed',
            results=[
                OutputOpenAIResponseOutputMessageFileSearchToolCallResult(
                    attributes={
                        'file_id': 'file-2a7bfe16e1724e35b54ffa0b4bca9686',
                        'chunk_id': '59485e55-3167-ed46-94b2-93ba856fc55a',
                        'filename': 'return-policy.txt',
                        'document_id': 'cb5d416e-3329-4e4e-8a65-45e93d755b54',
                        'token_count': 400.0,
                        'chunk_tokenizer': 'tiktoken:cl100k_base',
                        'metadata_token_count': 68.0
                    },
                    file_id='cb5d416e-3329-4e4e-8a65-45e93d755b54',
                    filename='cb5d416e-3329-4e4e-8a65-45e93d755b54',
                    score=13.334135130269885,
                    text='TechMart Return and Refund Policy\nShipping Information:\n- Standard Shipping: 3-5 
business days (Free on orders over $50)\n- Express Shipping: 1-2 business days ($15.99)\n- Overnight Shipping: Next
business day ($29.99, order before 2 PM EST)\n- Orders are processed within 24 hours on business days\n- Tracking 
number sent via email once shipped\nReturn Time Limits:\n- Standard items can be returned within 30 days of 
delivery\n- Electronics must be returned within 15 days of delivery\n- Opened software and personalized items 
cannot be returned\nReturn Conditions:\nItems must be in original condition with original packaging intact. All 
accessories, manuals, and tags must be included. Items showing signs of use may receive partial refund or be 
rejected.\nHow to Return an Item:\n1. Log into your TechMart account\n2. Go to My Orders and select the order\n3. 
Click Return Item and choose a reason\n4. You will receive a return label via email within 24 hours\n5. Pack the 
item securely and attach the return label\n6. Drop off at any carrier location\n7. Refund will be processed within 
5-7 business days after we receive the item\nRefund Amounts:\n- Full refund: Item returned within time limit, 
original condition, all components included\n- Partial refund: Missing accessories (20% deduction), damaged 
packaging (10% deduction), signs of use (15-30% deduction)\n- No refund: Returned after time limit, significantly 
damaged, missing major components\nDefective or Wrong Items:\nIf you receive a defective item or wrong item, we 
will provide a full replacement within 90 days at no cost. Return shipping is free and there is no restocking fee. 
Contact customer service immediately.\nRestocking Fees:\n- Unopened items: No restocking fee\n- Opened standard 
items: 10% restocking fee\n- Opened electronics: '
                ),
                OutputOpenAIResponseOutputMessageFileSearchToolCallResult(
                    attributes={
                        'file_id': 'file-2a7bfe16e1724e35b54ffa0b4bca9686',
                        'chunk_id': '20b6b53c-5ce6-8760-1832-6564cd41a474',
                        'filename': 'return-policy.txt',
                        'document_id': 'cb5d416e-3329-4e4e-8a65-45e93d755b54',
                        'token_count': 400.0,
                        'chunk_tokenizer': 'tiktoken:cl100k_base',
                        'metadata_token_count': 68.0
                    },
                    file_id='cb5d416e-3329-4e4e-8a65-45e93d755b54',
                    filename='cb5d416e-3329-4e4e-8a65-45e93d755b54',
                    score=12.097367160147689,
                    text='30% deduction)\n- No refund: Returned after time limit, significantly damaged, missing 
major components\nDefective or Wrong Items:\nIf you receive a defective item or wro

In [23]:
# this works with deployed granite embedding model with 
# - --hf-overrides
# '{"is_matryoshka":true,"matryoshka_dimensions":[768]}'
response = client.embeddings.create(
    input="Hello world",
    model="vllm-embedding/granite-embeddings",
    dimensions= "768"  
)

In [24]:
print(len(response.data[0].embedding))

768


In [25]:
rich.print(response)

CreateEmbeddingsResponse(
    data=[
        Data(
            embedding=[
                -0.01615055836737156,
                0.0010588907171040773,
                -0.011717071756720543,
                0.015517203137278557,
                -0.025967564433813095,
                -0.02612590231001377,
                0.01591305062174797,
                -0.019950689747929573,
                0.0016922459471970797,
                0.022800788283348083,
                -0.010292022489011288,
                -0.034992873668670654,
                -0.037842974066734314,
                -0.015279694460332394,
                -0.013537967577576637,
                -0.021375738084316254,
                -0.004453278612345457,
                0.01670474372804165,
                0.027392612770199776,
                -0.029451018199324608,
                0.03230111673474312,
                -0.06618562340736389,
                -0.034992873668670654,
                -0.04623493179678917,
                -0.03040105104446411,
                -0.017654776573181152,
                0.024225836619734764,
                -0.0134587986394763,
                -0.009658667258918285,
                0.03562622889876366,
                -0.019158994778990746,
                -0.01361713744699955,
                0.039901379495859146,
                0.016388066112995148,
                0.039901379495859146,
                -0.0340428426861763,
                -0.010054513812065125,
                -0.025650886818766594,
                -0.0015635957242920995,
                -0.02580922469496727,
                -0.011717071756720543,
                -0.0340428426861763,
                -0.025334209203720093,
                -0.02549254707992077,
                -0.007441923953592777,
                0.010450361296534538,
                0.00037605466786772013,
                -0.009975344873964787,
                -0.04591825231909752,
                0.03863466903567314,
                0.04116808995604515,
                0.0009500328451395035,
                -0.0030876067467033863,
                -0.004136601462960243,
                0.018050624057650566,
                0.0022860164754092693,
                0.031667761504650116,
                -0.019396502524614334,
                -0.014725509099662304,
                -0.005343934521079063,
                -0.02438417635858059,
                0.04401818662881851,
                -0.02897600084543228,
                -0.018921487033367157,
                0.014012984000146389,
                0.04401818662881851,
                0.01670474372804165,
                -0.011004546657204628,
                -0.04845167323946953,
                0.03087606653571129,
                0.021217400208115578,
                -0.02185075543820858,
                -0.031351082026958466,
                0.0025532131548970938,
                0.002454251516610384,
                -0.017179761081933975,
                -0.05795200169086456,
                -0.03483453765511513,
                0.005007464438676834,
                0.01725892908871174,
                0.008154448121786118,
                0.0137754762545228,
                -0.007362754549831152,
                0.03594290837645531,
                -0.03261779248714447,
                -0.023434143513441086,
                0.00042553554521873593,
                -0.04116808995604515,
                -0.019158994778990746,
                -0.021692415699362755,
                -0.010846207849681377,
                -0.005343934521079063,
                0.035309553146362305,
                0.007956525310873985,
                -0.028817662969231606,
                -0.023909159004688263,
                -0.028500985354185104,
                0.03356782719492912,
                -0.02612590231001377,
                0.0039584701880812645,
                0.0023750821128487587,
                -0.009896175004541874,
             